# File Validation & Routing Framework — Consolidated Design (v1.3, event-driven)

**Objective:** fetch files from four source systems, validate them against metadata-driven contracts (file type, size, emptiness, duplication, schema match, column order, optional regional/country residency), route PASS → Intake volume and FAIL → Quarantine, and prove end-to-end resilience by reconciling uploaded files against Databricks bronze processing.

## Sources

| # | Source | Arrival pattern | Detection |
| --- | --- | --- | --- |
| 1 | UI Portal (React) | Push, user-triggered | Direct upload API |
| 2 | SharePoint document libraries | Unannounced | Microsoft Graph change notifications (webhook), with change-token poller fallback |
| 3 | External → Internal SFTP | Unannounced | Internal-SFTP file-system watcher; poller with watermark + "file stable" check (size unchanged across two polls) fallback |
| 4 | Databricks Volume | Unannounced | Scheduled watcher diffing directory listing vs `file_registry` |

## Guiding principles

* **Metadata-driven** — file types, schemas, rules, routing, and retention are documents, not code.
* **Versioned contracts** — every result references the exact schema/rule version that produced it.
* **Canonical events** — every arrival, transition, and outcome is a normalized event; consumers never care which source produced it.
* **Idempotent everywhere** — at-least-once delivery + checksum/file-id dedup.
* **Events make it fast, reconciliation makes it provable.**

## Stack

React.js, FastAPI, Python; MongoDB = metadata & control plane only (bytes never rest in Mongo); file bytes stream directly to Databricks UC volumes via the Files API (storage v1.3, 2026-11-09); downstream Auto Loader, bronze/silver/gold.

## Reference Architecture

```text
┌─────────────┐ ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
│  UI Portal  │ │  SharePoint  │ │ Ext→Int SFTP│ │ Databricks  │
│  (React)    │ │ doc libraries│ │   relay     │ │   Volume    │
│ push, user- │ │ Graph webhook│ │  watcher    │ │  watcher    │
│  triggered  │ │              │ │             │ │             │
└──────┬──────┘ └──────┬──────┘ └──────┬──────┘ └──────┬──────┘
       └────────────┬──┴──────────────┴─┬─────────────┬┘
                    ▼                   ▼             ▼
     ┌──────────────────────────────────────────────┐
     │  EVENT INGESTION LAYER                       │
     │  FastAPI adapters + event_log                 │
     │  canonical FILE_ARRIVED event                 │
     │  (unique event_type + dedup_key → idempotent) │
     └───────────────────────┬──────────────────────┘
                             ▼
     STAGE → UC staging volume (Files API stream) + file_registry
     (upload context captured: user, IP,
      machine, session)
                             ▼
     IDENTIFY file type
     (explicit → filename pattern → header
      fingerprint → content sniff)
                             ▼
     VALIDATE — FILE → HEADER → CONTENT rules
     from rules registry; plugin validators;
     WARN = shadow mode
                             ▼
     ROUTE — routing_policies priority first-match
     → file_transfers manifest → Intake/Quarantine
     volume paths, checksum verified post-copy
                             ▼
     ┌──────────────────────────────────────────────┐
     │ Databricks Intake volume                     │
     │ per-file-type subfolders: customer, order,   │
     │ payments, product, employee, ...               │
     └───────────────────────┬──────────────────────┘
                             ▼
     Auto Loader → bronze → silver → gold
     (silver DQ exceptions → silver quarantine)
                             ▼
     INGESTION-RESULTS PUSHBACK job
     (per-file bronze stats → FastAPI →
      MongoDB databricks_ingestion_results)
                             ▼
     RECONCILIATION
     (uploaded = routed = transferred =
      bronze processed) → resilience certificate
     → gates staging purge (archival)
```

## Event-Driven Ingestion Architecture

### Hybrid detection strategy

| Source | Primary detection | Fallback | Operational notes |
| --- | --- | --- | --- |
| UI Portal (React) | Push — direct FastAPI upload API | n/a | Dedup by upload session; multi-file upload = one `upload_sessions` doc |
| SharePoint | Microsoft Graph change notifications (webhook) | Change-token poller | Subscription renewal before expiry + `validationToken` verification on every notification |
| External → Internal SFTP | Internal-SFTP file-system watcher | Poller with watermark + **"file stable" check** (size unchanged across two polls) | Stability check avoids reading partially-written files |
| Databricks Volume | Scheduled watcher diffing directory listing vs `file_registry` | Same (interval ~5 min) | Diff against `file_registry` self-heals missed arrivals |

### Canonical FILE_ARRIVED event

```json
{
  "event_id": "evt_01HV3...",
  "event_type": "FILE_ARRIVED",
  "source_id": "src_sftp_int",
  "detection_method": "WATCHER",
  "dedup_key": "src_sftp_int|/inbound/cust_master_20261106.csv|1762406400",
  "payload": {
    "uri": "sftp://internal/inbound/cust_master_20261106.csv",
    "size_bytes": 1048576,
    "mtime": "2026-11-06T08:00:00Z",
    "uploaded_by": "svc_sftp_acct",
    "client_ip": "10.2.3.4",
    "machine_name": "upstream-host-1",
    "upload_session_id": null
  },
  "received_at": "2026-11-06T08:00:12Z",
  "processing_status": "RECEIVED"
}
```

Portal events use `dedup_key = session_id|filename|part_index` (v1.3 single-PUT profile — `part_index` is reserved for future chunked sessions). The **unique index on `(event_type, dedup_key)`** makes duplicate deliveries harmless.

### Transport options

| Transport | When to use | Notes |
| --- | --- | --- |
| **MongoDB change streams on `event_log`** | **Recommended start** | Zero new infra; fits the MongoDB-centric stack. Adapters write events with `processing_status: RECEIVED`; FastAPI workers tail the change stream. Requires a replica set (Atlas or self-managed). |
| RabbitMQ / Redis Streams | Beyond a few thousand events/min | Competing consumers, native DLQ semantics |
| Kafka | Only if the org already runs it | Multi-consumer fan-out at scale |

### Event catalog

| Event | Emitted when | Consumer action |
| --- | --- | --- |
| `FILE_ARRIVED` | Adapter detects a new file | Stage to the UC staging volume (Files API PUT, v1.3), create `file_registry`, store upload context |
| `FILE_STAGED` | Checksum computed | Triggers identification |
| `FILE_IDENTIFIED` / `FILE_UNIDENTIFIED` | File type resolved / nothing matched | Starts validation / opens manual re-classification task |
| `VALIDATION_COMPLETED` | All bound rules executed | Triggers routing evaluation |
| `FILE_ROUTED` | Policy matched | Triggers `file_transfers` execution |
| `TRANSFER_COMPLETED` / `TRANSFER_FAILED` | Copy finished / failed | Marks `ROUTED_INTAKE` / opens retry + alert |
| `BRONZE_PROCESSED` | Pushback reports bronze stats | Feeds reconciliation |
| `RECONCILIATION_COMPLETED` | Reconciliation run finishes | Gates archival purge; emits resilience KPI |
| `FILE_ARCHIVED` / `FILE_PURGED` | Retention job acts | Updates `file_lifecycle` |

### Reliability semantics

* **At-least-once delivery with idempotent consumers** — `file_registry` upsert by checksum; `event_log` dedup via the unique `(event_type, dedup_key)` index.
* **Retries** — 3× exponential backoff, then `processing_status: DEAD`.
* **DEAD events** — listed in the portal ops view with a **re-drive action**; alert on the first DEAD event per source per hour.
* **Ordering is not required** — the state machine rejects illegal transitions, so out-of-order or replayed events cannot corrupt state.
* **Missed-event safety net** — scheduled SFTP/volume watchers diff against `file_registry` (self-healing for missed webhooks), and reconciliation catches anything after Intake.

> **Events reduce latency; watchers + reconciliation guarantee correctness.**

## Physical Storage (v1.3 — Direct-to-Volume; GridFS deprecated)

**Decision (Storage Addendum v1.3, 2026-11-09):** file bytes never rest in MongoDB. FastAPI streams them directly to UC volumes via the Databricks Files API (`PUT/GET/HEAD/DELETE /api/2.0/fs/files/Volumes/...`) with M2M OAuth; MongoDB is the metadata + control plane only. `file_registry.staged_path` is the primary bytes pointer; `gridfs_file_id` is nullable/legacy.

* Staging roots per source with per-session subfolders; sha256 uniqueness checked at finalize.
* Small-file profile (< 500 MB): single streaming PUT with on-the-fly size cap; idempotent re-PUT (same path, overwrite).
* Routing transfer: worker GETs from the staging volume, re-hashes, PUTs to intake/quarantine, HEAD-verifies, then deletes the staging copy.
* Purge tombstone `volume_purged_at`; retention gate unchanged — purge only after `MATCHED` reconciliation or legal-hold override.

### Alternative considered — BSON-embedded (BinData) staging (evaluated 2026-11-10)

MongoDB documents can hold binary up to ~16 MB, so sub-15 MB files can be stored as-is as `BinData` in a normal collection — no chunking, no GridFS, and the bytes + metadata land in one atomic document write. This option was evaluated against direct-to-volume:

| Dimension | BSON-embedded | Direct-to-volume (chosen) |
| --- | --- | --- |
| Staging write | One atomic doc write (bytes + metadata together) | Volume PUT (Files API stream) + Mongo metadata in the finalize guard transaction |
| Path to intake | Worker must pull the full doc from Mongo, then re-PUT to the volume — every byte crosses the network twice | Worker reads the staging volume once; Mongo never holds bytes |
| Replication & change streams | 15 MB docs inflate oplog entries → replication lag, bigger change-stream payloads | Oplog carries only small metadata docs |
| Storage cost & cache | Atlas GB ≈ several × volume storage; WiredTiger cache pressure unless bytes live in a dedicated `file_blobs` collection | Bytes on cheap volume storage; Mongo working set stays small |
| Size ceiling | Hard 16 MB BSON cap — every file 15–500 MB needs a second storage path forever | One threshold: 500 MB manifest cap; chunked sessions later |
| Ack semantics | Ack after the doc insert | Ack (finalize) after the durable volume write is HEAD-verified + guard transaction recorded |
| Purge / retention | Byte purge = doc delete; retention collides with backup/oplog windows | `volume_purged_at` tombstone + object delete |

**Verdict:** a legitimate pattern (and the simplest single-write case), but it turns MongoDB into a hot byte store — double transfer on every routed file, oplog/cache pressure, a permanent two-profile size ceiling, and Atlas storage costs — while direct-to-volume delivers the same no-chunking simplicity with bytes written once. **Chosen: direct-to-volume.** BSON-embedded is recorded as the fallback profile if the API tier ever loses direct volume access (all files provably < 15 MB).

### Acknowledgement → Auto Loader semantics

Auto Loader needs **no ack to start** — a streaming read against the intake volume (directory listing or file-notification mode) picks up routed files autonomously. The framework's acknowledgements serve different consumers:

1. **Upload ack** — `finalize` returns only after the staging volume write is durable and HEAD-verified, with the `file_registry` + `file_lifecycle` + `event_log` guard transaction recorded; the client may then flush its local copy.
2. **Routing ack** — `TRANSFER_COMPLETED` + `file_lifecycle: INTAKE_VOLUME` mark the file as visible in intake (drives portal UI states).
3. **Processing ack** — bronze pushback writes `databricks_ingestion_results`; reconciliation certifies `MATCHED` (the resilience KPI).

If trigger-on-ack is ever required (start Databricks processing only after routing acks), a streaming table over the intake volume with a Databricks-side watch on `event_log` push notifications is the extension point — currently deferred as the post-R2 validation option.

## Collection Catalog (21)

> **15 of these are applied in Phase 1 dev** (Steps 1–2, per the Deployment Runbook). `retention_policies`, `batches`, `archive_jobs`, `databricks_ingestion_results`, `reconciliation_runs`, `reconciliation_results` land with Phases 2–3.

### Registry layer (8) — identity + rules-as-data

| Collection | Key fields |
| --- | --- |
| `users` | `user_id`, `email` (unique), `display_name`, `roles[]` (UPLOADER / OPS / ADMIN), `department`, `is_active`, `created_at`, `last_login_at` |
| `source_systems` | `source_id`, `name`, `source_type` (PORTAL / SHAREPOINT / SFTP / DB_VOLUME), `connection_ref`, `poll_or_push`, `detection_config`, `is_active` |
| `file_types` | `file_type_id`, `code`, `allowed_extensions[]`, `max_file_size_mb`, `min_rows`, `encoding`, `delimiter`, `header_row`, `identification` {`patterns`, `header_fingerprint`}, `is_active`, `version` |
| `schema_versions` | `schema_id`, `file_type_id`, `version`, `status` (draft / active / retired), `effective_from`, `fields[]` {`order`, `name`, `data_type`, `nullable`, `max_length`, `format`, `is_pii`, `pii_classification`} |
| `validation_rules` | `rule_id`, `rule_code`, `category` (FILE / HEADER / CONTENT), `severity` (ERROR / WARN), `applies_to`, `params` (json), `priority`, `is_active`, `version`, `created_by` |
| `routing_policies` | `policy_id`, `priority`, `match` (json), `action` (MOVE_INTAKE / MOVE_QUARANTINE / REJECT / NOTIFY_ONLY), `is_active` |
| `volume_map` | `file_type_id`, `destination_type`, `volume_root`, `subfolder`, `naming_convention` (default `{file_type}_{file_id}_{ts}.ext`) |
| `retention_policies` | `policy_id`, `file_type_id` (or `*` for all), `status_class` (INTAKE_COPY / QUARANTINED / UNIDENTIFIED / ALL), `retention_days`, `archive_after_days`, `archive_target`, `purge_after_days`, `legal_hold` |

### Eventing layer (1)

| Collection | Key fields |
| --- | --- |
| `event_log` | Canonical event store + queue: fields per the Event-Driven Ingestion section (`event_id`, `event_type`, `source_id`, `detection_method`, `dedup_key`, `payload`, `received_at`) plus `processing_status` (RECEIVED / PROCESSING / PROCESSED / DEAD), `retry_count`, `last_error`, `correlation_id` |

### Execution layer (5)

| Collection | Key fields |
| --- | --- |
| `upload_sessions` | `session_id`, `source_id`, `user_id`, `user_email`, `client_ip`, `machine_name`, `user_agent`, `auth_method`, `roles[]`, `files_declared_count`, `files_received_count`, `started_at`, `completed_at` |
| `file_registry` | **Central control entity + state machine**: `file_id`, `source_id`, `original_name`, `staged_path` (primary bytes pointer, v1.3), `gridfs_file_id` (nullable, legacy), `size_bytes`, `extension`, `mime_type`, `sha256_checksum`, `file_type_id`, `schema_version_id`, `identification_method`, `identification_confidence`, `identification_evidence`, `unidentified_reason`, `upload_session_id`, `uploaded_by`, `client_ip`, `machine_name`, `current_status`, `batch_id`, `retry_count`, `received_at`, `updated_at`, `volume_purged_at` (purge tombstone, v1.3) |
| `validation_results` | `file_id`, `batch_id`, `rule_id`, `rule_version`, `status` (PASS / FAIL / WARN / SKIPPED / ERROR), `details` (json evidence), `executed_at`, `duration_ms` |
| `batches` | `batch_id`, `source_id`, `triggered_by` (event / schedule / user), `watermark`, `status`, `files_total` / `passed` / `failed` / `quarantined`, `started_at`, `ended_at` |
| `file_transfers` | **Routing manifest**: `transfer_id`, `file_id`, `policy_id`, `from_location` (`volume:<staging_path>`), `to_location` (resolved volume path), `direction` (MONGO_TO_INTAKE / MONGO_TO_QUARANTINE / MONGO_TO_ARCHIVE — legacy labels kept in v1.3 to match the Step-2 DDL and worker; the storage change is carried by the `volume:` location prefixes), `status` (PENDING / IN_PROGRESS / COMPLETED / FAILED / RETRYING), `attempts`, `last_error`, `source_sha256`, `dest_sha256`, `bytes_written`, `worker_id`, `started_at`, `completed_at` |

### Governance layer (4)

| Collection | Key fields |
| --- | --- |
| `audit_log` | Append-only: `event_id`, `ts`, `actor`, `entity_type`, `entity_id`, `action` (CONFIG_CHANGE / STATUS_CHANGE / FILE_ROUTED / RULE_ADDED), `before`, `after`, `correlation_id`, `approver` |
| `file_lifecycle` | `file_id`, `location` (STAGING_VOLUME / INTAKE_VOLUME / QUARANTINE / ARCHIVE / PURGED; GRIDFS retained as legacy value), `timestamp`, `triggered_by_job` |
| `notifications` | `notification_id`, `event`, `condition` (json), `channel` (email / teams / webhook), `recipients`, `is_active` |
| `archive_jobs` | `job_id`, `policy_id`, `status`, `files_scanned` / `archived` / `purged`, `started_at`, `ended_at` |

### Resilience layer (3)

| Collection | Key fields |
| --- | --- |
| `databricks_ingestion_results` | `ingestion_id`, `run_id`, `bronze_table`, `file_type_id`, `file_id`, `sha256`, `filename`, `rows_read`, `rows_written`, `rows_quarantined_silver`, `dq_results` (json), `processed_at` |
| `reconciliation_runs` | `run_id`, `scope` (batch / daily / file_type), `expected_files`, `matched`, `discrepancies`, `status`, `started_at`, `ended_at` |
| `reconciliation_results` | `file_id`, `run_id`, `uploaded_flag`, `routed_flag`, `transfer_sha_match`, `bronze_processed_flag`, `row_delta`, `status` (MATCHED / MISSING_IN_BRONZE / ROW_MISMATCH / DUPLICATE_PROCESSED / STUCK_IN_TRANSIT), `severity`, `resolution_status` |

## Data Dictionary (1/3) — Registry Layer (Build Step 1)

**How to read this dictionary** (applies to all three parts):

| Relational concept | In this MongoDB design |
| --- | --- |
| Table | Collection (15 applied in Phase-1 dev; 6 more land with Phases 2–3) |
| Primary key | Every collection has the auto `_id` (ObjectId, 12 bytes); the **logical PK is a business key with a unique index** (shown per collection) |
| Foreign key | **Logical reference only** — Mongo does not enforce FKs; integrity is guarded by `$jsonSchema` validators (strict/error), the API state machine, and unique indexes |
| Size | BSON constraints per field (pattern / minLength / maxLength / min-max); every document is KB-scale vs the 16 MB BSON cap — **file bytes never rest in Mongo** (storage v1.3) |
| Layer | Registry = rules-as-data config • Eventing = canonical queue • Execution = runtime state • Governance = append-only trail • Resilience = proof of correctness |

All collections use `validationLevel: strict`, `validationAction: error`.

#### `users` — Layer: Registry • Use: portal identity + RBAC

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `user_id` | string | no | `^usr_` | unique user identifier — **logical PK** |
| `email` | string | no | minLength 5, unique | login + contact |
| `display_name` | string | no | minLength 1 | UI label |
| `roles` | array\<string\> | no | minItems 1; enum UPLOADER / OPS / ADMIN | RBAC roles |
| `department` | string | yes | — | org unit |
| `is_active` | bool | no | — | soft-disable flag |
| `created_at` | date | no | — | provisioning time |
| `last_login_at` | date | yes | — | auth telemetry |

**PK** `user_id` • **FK** — (referenced by `upload_sessions.user_id`, `audit_log.actor`) • **Indexes** `{user_id}` unique, `{email}` unique • **Seeds** bootstrap admin (set via portal/mongosh, not seeded).

#### `source_systems` — Registry • ingestion source registration

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `source_id` | string | no | `^src_` | **logical PK** |
| `name` | string | no | minLength 1 | display name |
| `source_type` | enum | no | PORTAL / SHAREPOINT / SFTP / DB_VOLUME | arrival channel |
| `connection_ref` | string | no | minLength 1 | vault reference — never a literal credential |
| `poll_or_push` | enum | no | PUSH / POLL | detection pattern |
| `detection_config` | object | yes | — | per-source config (webhook IDs, watch paths, watermark) |
| `is_active` | bool | no | — | enable/disable channel |

**PK** `source_id` • **FK** — (referenced by `event_log` / `upload_sessions` / `file_registry` `.source_id`) • **Indexes** `{source_id}` unique • **Seeds** 1 (`src_portal`).

#### `file_types` — Registry • file-type contracts

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `file_type_id` | string | no | `^ft_` | **logical PK** |
| `code` | string | no | minLength 2, unique | short code (customer, order…) |
| `description` | string | yes | — | human description |
| `allowed_extensions` | array\<string\> | no | minItems 1; items `^\.` | e.g. `.csv` — feeds FILE_TYPE_ALLOWED |
| `max_file_size_mb` | int | yes | min 1 | feeds FILE_SIZE_MAX |
| `min_rows` | int | yes | min 0 | feeds FILE_NOT_EMPTY_ROWS |
| `encoding` | string | yes | — | e.g. `utf-8` (worker decode) |
| `delimiter` | string | yes | — | header parsing |
| `header_row` | bool | no | — | HEADER_REQUIRED |
| `identification` | object | yes | — | {`patterns[]` (regex), `header_fingerprint[]`} — identification cascade |
| `is_active` | bool | no | — | — |
| `version` | int | no | min 1 | contract version |

**PK** `file_type_id` • **FK** — (referenced by `schema_versions`, `volume_map`, `file_registry` `.file_type_id`) • **Indexes** `{file_type_id}` unique, `{code}` unique • **Seeds** 5 (customer, order, payments, product, employee).

#### `schema_versions` — Registry • column contracts (versioned)

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `schema_id` | string | no | `^sch_` | **logical PK** |
| `file_type_id` | string | no | `^ft_` | **FK →** `file_types` |
| `version` | int | no | min 1 | per-type version |
| `status` | enum | no | draft / active / retired | lifecycle state |
| `effective_from` | date | no | — | activation time |
| `fields` | array\<object\> | no | minItems 1 | items: `order` int≥1, `name` string, `data_type` enum (string/integer/decimal/date/boolean/timestamp), `nullable` bool, `max_length` int, `format` string, `is_pii` bool, `pii_classification` string — the ordered column contract |
| `created_by` | string | no | — | user or system id |
| `created_at` | date | no | — | — |

**PK** `schema_id` • **FK** `file_type_id` → `file_types` • **Indexes** `{schema_id}` unique; `{file_type_id, version}` unique; `{file_type_id}` **unique, partial WHERE status = "active"** — exactly one active version per file type • **Seeds** 5 (`sch_*_v1`).

#### `validation_rules` — Registry • rules-as-data

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `rule_id` | string | no | `^rule_` | **logical PK** |
| `rule_code` | string | no | minLength 3 | plugin key (HEADER_REQUIRED…) |
| `category` | enum | no | FILE / HEADER / CONTENT | execution phase |
| `severity` | enum | no | ERROR / WARN | WARN = shadow mode |
| `applies_to` | string | yes | — | file-type filter (null = all) |
| `params` | object | no | — | rule parameters (e.g. `min_rows`, `max_mb`) |
| `priority` | int | no | min 1 | execution order within category |
| `is_active` | bool | no | — | — |
| `version` | int | no | min 1 | rule version (result evidence refs it) |
| `created_by` | string | no | — | — |
| `created_at` | date | no | — | — |
| `updated_at` | date | yes | — | last edit |

**PK** `rule_id` • **FK** — (referenced by `validation_results.rule_id`) • **Indexes** `{rule_id}` unique; `{rule_code, version}` unique • **Seeds** 8 (FILE×4, HEADER×3, CONTENT×1).

#### `routing_policies` — Registry • first-match routing

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `policy_id` | string | no | `^pol_` | **logical PK** |
| `name` | string | no | minLength 1 | display name |
| `priority` | int | no | min 1 | scan order (asc) — first match wins |
| `match` | object | no | — | json conditions (any_error, status…) |
| `action` | enum | no | MOVE_INTAKE / MOVE_QUARANTINE / REJECT / NOTIFY_ONLY | destination decision |
| `is_active` | bool | no | — | — |

**PK** `policy_id` • **FK** — (referenced by `file_transfers.policy_id`) • **Indexes** `{policy_id}` unique; `{priority}` • **Seeds** 3 (error→quarantine, unidentified→quarantine, passed/warned→intake) • no-match fail-safe = quarantine.

#### `volume_map` — Registry • destination resolution

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `map_id` | string | no | `^vol_` | **logical PK** |
| `file_type_id` | string | no | — | **FK →** `file_types` (`*` = catch-all) |
| `destination_type` | enum | no | STAGING / INTAKE / QUARANTINE / REJECT | target volume tier |
| `volume_root` | string | no | minLength 1 | absolute `/Volumes/mdd_platform/fvr/…` path |
| `subfolder` | string | no | — | per-type folder (customer, order, payments…) |
| `naming_convention` | string | no | minLength 1 | `{file_type}_{file_id}_{ts}.ext` / `{original_name}` |

**PK** `map_id` • **FK** `file_type_id` → `file_types` (`*` wildcard) • **Indexes** `{map_id}` unique; `{file_type_id, destination_type}` unique • **Seeds** 7 (5 intake + quarantine catch-all + staging catch-all).

## Data Dictionary (2/3) — Eventing & Execution Layer (Build Step 2)

#### `event_log` — Layer: Eventing • Use: canonical event store + work queue

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `event_id` | string | no | `^evt_` | **logical PK** |
| `event_type` | enum | no | FILE_ARRIVED / FILE_STAGED / FILE_IDENTIFIED / FILE_UNIDENTIFIED / VALIDATION_COMPLETED / FILE_ROUTED / TRANSFER_COMPLETED / TRANSFER_FAILED / BRONZE_PROCESSED / RECONCILIATION_COMPLETED / FILE_ARCHIVED / FILE_PURGED | canonical event kind |
| `source_id` | string | no | — | **FK →** `source_systems` |
| `detection_method` | enum | no | PUSH / WEBHOOK / WATCHER / POLL | how detected |
| `dedup_key` | string | no | minLength 1 | idempotency key (source|uri|mtime, session|filename|part) |
| `payload` | object | no | — | event body (KB-scale; uri, size_bytes, uploaded_by…) |
| `processing_status` | enum | no | RECEIVED / PROCESSING / PROCESSED / DEAD | queue state |
| `retry_count` | int | no | min 0 | retry attempts |
| `last_error` | string | yes | — | last failure reason |
| `correlation_id` | string | yes | — | cross-system trace |
| `received_at` | date | no | — | ingestion time |

**PK** `event_id` • **Structural guard: `{event_type, dedup_key}` unique — idempotent ingestion, replays harmless** • **FK** `source_id` → `source_systems`; payload carries `file_id` refs • **Indexes** `{event_id}` unique; `{event_type, dedup_key}` unique; `{processing_status, received_at}` (worker scan + queue depth); `{received_at}` partial WHERE DEAD (re-drive view) • **TTL** 30 d on PROCESSED — commented out pending compliance sign-off • **Seeds** none (written by adapters/API/worker only).

#### `upload_sessions` — Execution • portal upload interaction record

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `session_id` | string | no | `^ses_` | **logical PK** |
| `source_id` | string | no | — | **FK →** `source_systems` (always `src_portal`) |
| `status` | enum | no | IN_PROGRESS / COMPLETED / COMPLETED_PARTIAL / INCOMPLETE / ABANDONED | SESSION_STATUS machine |
| `user_id` | string | no | — | **FK →** `users` |
| `user_email` | string | no | — | uploader email |
| `client_ip` | string | no | — | provenance |
| `machine_name` | string | yes | — | provenance |
| `user_agent` | string | yes | — | provenance |
| `auth_method` | string | yes | — | provenance |
| `roles_snapshot` | array\<string\> | no | minItems 1; enum UPLOADER / OPS / ADMIN | RBAC snapshot at upload time |
| `files_declared_count` | int | yes | min 0 | manifest declaration |
| `files_received_count` | int | no | min 0 | actual received |
| `parts` | array\<object\> | no | items: `file_id` (string, **FK →** `file_registry`), `staged_path` (string|null), `status` (enum UPLOADING / FINALIZED / ABANDONED), `bytes_received` (long, min 0), `sha256` (string|null, maxLength 64) | per-file upload state |
| `started_at` | date | no | — | — |
| `completed_at` | date | yes | — | — |

**PK** `session_id` • **FK** `source_id`, `user_id` → `users`, `parts[].file_id` → `file_registry` • **Indexes** `{session_id}` unique; `{user_id, started_at desc}` (my-uploads); `{status, started_at}` (session janitor) • **Seeds** none.

#### `file_registry` — Execution • **central control entity + state machine** (the spine)

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `file_id` | string | no | `^file_` | **logical PK** |
| `source_id` | string | no | — | **FK →** `source_systems` |
| `original_name` | string | no | minLength 1 | name as uploaded |
| `staged_path` | string | yes | — | **primary bytes pointer** (storage v1.3): `/Volumes/…/staging/sessions/<ses_id>/<name>` |
| `gridfs_file_id` | objectId | yes | — | **legacy** — null for all Phase-1 files |
| `size_bytes` | long / int | no | min 0 | manifest-verified size |
| `extension` | string | no | — | normalized extension |
| `mime_type` | string | yes | — | declared mime |
| `sha256_checksum` | string | no | exactly 64 hex | **unique — duplicate guard (C1)** |
| `file_type_id` | string | yes | — | **FK →** `file_types` |
| `schema_version_id` | string | yes | — | **FK →** `schema_versions` (contract used) |
| `identification_method` | enum | yes | EXPLICIT / FILENAME_PATTERN / HEADER_FINGERPRINT / CONTENT_SNIFF | how typed |
| `identification_confidence` | double / int | yes | — | 0–1 score |
| `identification_evidence` | object | yes | — | match details |
| `unidentified_reason` | string | yes | — | why UNIDENTIFIED |
| `upload_session_id` | string | yes | — | **FK →** `upload_sessions` |
| `uploaded_by` | string | yes | — | provenance |
| `client_ip` | string | yes | — | provenance |
| `machine_name` | string | yes | — | provenance |
| `current_status` | enum | no | RECEIVED / IDENTIFYING / VALIDATING / PASSED / FAILED / WARNED / UNIDENTIFIED / ROUTED_INTAKE / ROUTED_QUARANTINE / REJECTED / FRAMEWORK_ERROR | state machine |
| `batch_id` | string | yes | — | Phase-2 grouping |
| `retry_count` | int | no | min 0 | worker retries |
| `received_at` | date | no | — | — |
| `updated_at` | date | no | — | — |
| `volume_purged_at` | date | yes | — | **purge tombstone** (v1.3) |
| `legal_hold` | bool | yes | — | per-file retention override |

**PK** `file_id` • **Structural guard: `{sha256_checksum}` unique** • **FKs** `source_id`, `file_type_id`, `schema_version_id`, `upload_session_id` • **Indexes** `{file_id}` unique; `{sha256_checksum}` unique; `{current_status}` (ops board); `{source_id, received_at desc}`; `{upload_session_id}`; `{file_type_id, current_status, received_at}` **partial WHERE volume_purged_at = null** (purge-candidate scan) • **Seeds** none (API-created only).

#### `validation_results` — Execution • rule evidence (append-only incl. re-validates)

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `result_id` | string | no | `^vr_` | **logical PK** |
| `file_id` | string | no | — | **FK →** `file_registry` |
| `batch_id` | string | yes | — | Phase-2 batch grouping |
| `run_seq` | int | no | min 1 | 1 = original run; 2, 3… = re-validates |
| `rule_id` | string | no | — | **FK →** `validation_rules` |
| `rule_version` | int | no | min 1 | versioned evidence |
| `status` | enum | no | PASS / FAIL / WARN / SKIPPED / ERROR | verdict |
| `details` | object | no | — | json evidence (expected/actual/missing/extra) |
| `executed_at` | date | no | — | — |
| `duration_ms` | int / long | yes | min 0 | timing |

**PK** `result_id` • **FKs** `file_id`, `rule_id`, `batch_id` (Phase 2) • **Indexes** `{result_id}` unique; `{file_id, run_seq, executed_at desc}` (results screen + latest run) • **Seeds** none.

#### `file_transfers` — Execution • routing manifest (one COMPLETED copy per direction)

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `transfer_id` | string | no | `^trf_` | **logical PK** |
| `file_id` | string | no | — | **FK →** `file_registry` |
| `policy_id` | string | yes | — | **FK →** `routing_policies` (policy that fired) |
| `from_location` | string | no | — | `volume:<staged_path>` |
| `to_location` | string | no | — | resolved destination path |
| `direction` | enum | no | MONGO_TO_INTAKE / MONGO_TO_QUARANTINE / MONGO_TO_ARCHIVE | legacy labels kept (match Step-2 DDL + worker); volume semantics carried by `volume:` prefixes |
| `status` | enum | no | PENDING / IN_PROGRESS / COMPLETED / FAILED / RETRYING | transfer machine |
| `attempts` | int | no | min 0 | capped at 3 then ops flag |
| `last_error` | string | yes | — | — |
| `source_sha256` | string | no | exactly 64 hex | pre-move hash |
| `dest_sha256` | string | yes | exactly 64 hex | post-move verify hash |
| `bytes_written` | long / int | yes | min 0 | — |
| `worker_id` | string | yes | — | atomic claim owner |
| `started_at` | date | yes | — | — |
| `completed_at` | date | yes | — | — |

**PK** `transfer_id` • **Structural guard: `{file_id, direction}` unique, partial WHERE status = COMPLETED — a file can never complete the same direction twice (C4)** • **FKs** `file_id`, `policy_id` • **Indexes** `{transfer_id}` unique; the partial-unique guard; `{status, started_at}` (retry scanner / backpressure) • **Seeds** none.

## Data Dictionary (3/3) — Governance & Resilience + FK Graph

#### `file_lifecycle` — Layer: Governance • append-only physical-location history

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `event_id` | string | no | `^flc_` | **logical PK** |
| `file_id` | string | no | — | **FK →** `file_registry` |
| `location` | enum | no | STAGING_VOLUME / INTAKE_VOLUME / QUARANTINE / ARCHIVE / PURGED (GRIDFS legacy) | where the bytes were |
| `timestamp` | date | no | — | — |
| `triggered_by_job` | string | yes | — | purge/archive job id |

**PK** `event_id` • **FK** `file_id` • **Indexes** `{event_id}` unique; `{file_id, timestamp}` • **Use:** where-have-the-bytes-been audit trail; drives purge-gate evidence. **Seeds** none.

#### `audit_log` — Governance • append-only human/system action trail

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `event_id` | string | no | `^aud_` | **logical PK** |
| `ts` | date | no | — | — |
| `actor` | string | no | minLength 1 | `user_id` or `system:<component>` |
| `entity_type` | string | no | — | e.g. `file_registry`, `validation_rules` |
| `entity_id` | string | no | — | logical FK to any collection |
| `action` | enum | no | CONFIG_CHANGE / STATUS_CHANGE / FILE_ROUTED / RULE_ADDED / FILE_PURGED / OTHER | — |
| `before` | object | yes | — | pre-change snapshot |
| `after` | object | yes | — | post-change snapshot |
| `approver` | string | yes | — | dual-control approver (config changes) |
| `correlation_id` | string | yes | — | trace link |

**PK** `event_id` • **FK** `entity_id` (polymorphic) • **Indexes** `{event_id}` unique; `{entity_id, ts desc}` (drill-down); `{ts}` (time-range pulls) • **Use:** governance; app writes via insert-only role `fvr_app_audit_writer`. **Seeds** none.

#### `notifications` — Governance • alert routing config

| Field | Type | Null | Size / constraint | Description |
| --- | --- | --- | --- | --- |
| `notification_id` | string | no | `^not_` | **logical PK** |
| `event` | string | no | minLength 1 | event type to alert on |
| `condition` | object | yes | — | e.g. `{min_attempts: 3}` |
| `channel` | enum | no | email / teams / webhook | delivery channel |
| `recipients` | array\<string\> | no | minItems 1 | targets |
| `is_active` | bool | no | — | — |

**PK** `notification_id` • **FK** — • **Indexes** `{notification_id}` unique; `{event, is_active}` • **Seeds** 3 (`not_transfer_failed`, `not_framework_error`, `not_session_incomplete`) — **recipients are placeholders, open item to edit**.

#### Phase 2–3 collections (design-level — DDL lands with their build steps)

| Collection | Layer | Key fields (logical PK / FKs) | Use |
| --- | --- | --- | --- |
| `databricks_ingestion_results` | Resilience | `ingestion_id` (PK), `run_id`, `file_id`, `sha256`, `rows_read` / `rows_written` / `rows_quarantined_silver`, `dq_results` | bronze pushback evidence per file |
| `reconciliation_runs` | Resilience | `run_id` (PK), `scope`, `expected_files`, `matched`, `discrepancies`, `status` | resilience certificate runs |
| `reconciliation_results` | Resilience | `file_id` + `run_id`, `uploaded_flag`, `routed_flag`, `transfer_sha_match`, `bronze_processed_flag`, `status` (MATCHED / MISSING_IN_BRONZE / ROW_MISMATCH / DUPLICATE_PROCESSED / STUCK_IN_TRANSIT) | per-file 4-way match |
| `batches` | Execution | `batch_id` (PK), `source_id`, `watermark`, `files_total` / `passed` / `failed` / `quarantined` | grouping + progress |
| `retention_policies` | Registry | `policy_id` (PK), `file_type_id` (`*`), `status_class`, `retention_days` / `archive_after_days` / `purge_after_days`, `legal_hold` | lifecycle configuration |
| `archive_jobs` | Governance | `job_id` (PK), `policy_id`, `files_scanned` / `archived` / `purged` | retention job runs |

### Logical FK graph (Phase 1)

```text
source_systems ─┬─< event_log, upload_sessions, file_registry  (.source_id)
users ──────────┴─< upload_sessions (.user_id)
file_types ─┬─< schema_versions (.file_type_id)
            ├─< volume_map (.file_type_id, `*` catch-all)
            └─< file_registry (.file_type_id)
schema_versions ──< file_registry (.schema_version_id)
upload_sessions ──< file_registry (.upload_session_id); parts[].file_id → file_registry
file_registry ─┬─< validation_results, file_transfers, file_lifecycle (.file_id)
               └──< audit_log (entity_id where entity_type = file_registry)
validation_rules ──< validation_results (.rule_id)
routing_policies ──< file_transfers (.policy_id)
event_log ⇄ all runtime collections via payload.file_id + correlation_id
```

**Integrity summary:** 15 Phase-1 collections, 5 layers, 3 structural unique guards (`event_type+dedup_key`, `sha256_checksum`, `file_id+direction WHERE COMPLETED`), 1 partial-unique invariant (one active `schema_version` per file type), and every write through `$jsonSchema` strict/error validation.

## File Status State Machine

```text
RECEIVED → IDENTIFYING → VALIDATING → PASSED  → ROUTED_INTAKE
                                 → WARNED  → ROUTED_INTAKE   (flagged)
                                 → FAILED  → ROUTED_QUARANTINE

RECEIVED → UNIDENTIFIED → (manual re-classification) → VALIDATING

any state → FRAMEWORK_ERROR → retry → previous state

REJECTED  (terminal)
```

Illegal transitions are rejected by the API layer and logged to `audit_log` — so out-of-order or replayed events can never corrupt state.

## File-Type Identification

Priority order — stop at first hit, store the evidence:

1. **Explicit** — portal user selection; SharePoint library ↔ file type; SFTP directory ↔ file type (**mandatory config** for SFTP/SharePoint mappings).
2. **Filename pattern** — regex per file type (`cust_master_YYYYMMDD.csv`).
3. **Header fingerprint** — ordered match of the header row against `schema_versions.fields[].name`.
4. **Content sniffing** — sampled-row scoring against the schema (lowest confidence).

Evidence + confidence are stored on `file_registry` (`identification_method`, `identification_confidence`, `identification_evidence`). Unidentified files route to the `UNIDENTIFIED` quarantine subfolder with a **manual re-classification UI action** that sets the file type, re-enters validation, and is audited.

## Rule Extensibility

1. **Parameter-only rules** — insert a new `validation_rules` doc reusing an existing `rule_code` with new `params`; no deploy. Covers ~80% of new rules.
2. **New rule kinds** — one Python class per `BaseValidator`, registered via a decorator keyed by `rule_code`. Unregistered codes found in config surface as `FRAMEWORK_ERROR` — config mistakes never pass silently.
3. **Shadow onboarding** — add the rule with severity `WARN`, observe `validation_results` for N batches, then flip to `ERROR`. No quarantines during tuning.

## Reconciliation ↔ Archival Coupling

* Reconciliation compares **uploaded vs routed vs transferred vs bronze-processed** per file, keyed by `file_id` / `sha256` propagated into bronze `_metadata`.
* A clean `MATCHED` across a batch is the **resilience certificate** — the headline portal KPI.
* **Retention purge gate** — staging purge (volume object delete + `volume_purged_at` tombstone) requires a `MATCHED` reconciliation result, or an explicit legal-hold override.

## MongoDB Index & Schema Validation Specs

> **Superseded by the Build Step 1–2 notebooks** (authoritative DDL — carries the LLD v1.1–v1.3 additions incl. storage v1.3, `users`, `run_seq`, `volume_purged_at`). Kept as the consolidated reference.

Run in `mongosh` against the `fvr` database. `$jsonSchema` validators use `validationLevel: "strict"` and `validationAction: "error"` for write-critical collections (use `"moderate"` where legacy documents must be tolerated).

### (a) `file_registry` — collection + validator

```javascript
use fvr

db.createCollection("file_registry", {
  validator: {
    $jsonSchema: {
      bsonType: "object",
      required: ["file_id", "source_id", "original_name", "staged_path",
                 "sha256_checksum", "size_bytes", "extension", "current_status", "received_at"],
      properties: {
        file_id:                 { bsonType: "string", pattern: "^file_" },
        source_id:               { bsonType: "string" },
        original_name:           { bsonType: "string" },
        staged_path:              { bsonType: "string" },           // primary bytes pointer (storage v1.3)
        gridfs_file_id:           { bsonType: ["objectId", "null"] },  // legacy only, nullable
        size_bytes:              { bsonType: ["long", "int"], minimum: 0 },
        extension:               { bsonType: "string" },
        mime_type:               { bsonType: "string" },
        sha256_checksum:         { bsonType: "string", minLength: 64, maxLength: 64 },
        file_type_id:            { bsonType: ["string", "null"] },
        schema_version_id:       { bsonType: ["string", "null"] },
        identification_method:   { enum: ["EXPLICIT", "FILENAME_PATTERN", "HEADER_FINGERPRINT", "CONTENT_SNIFF", null] },
        identification_confidence: { bsonType: ["double", "int", "null"] },
        current_status: { enum: ["RECEIVED", "IDENTIFYING", "VALIDATING", "PASSED", "FAILED",
                                "WARNED", "UNIDENTIFIED", "ROUTED_INTAKE", "ROUTED_QUARANTINE",
                                "REJECTED", "FRAMEWORK_ERROR"] },
        upload_session_id: { bsonType: ["string", "null"] },
        uploaded_by:       { bsonType: ["string", "null"] },
        client_ip:         { bsonType: ["string", "null"] },
        machine_name:     { bsonType: ["string", "null"] },
        batch_id:         { bsonType: ["string", "null"] },
        retry_count:      { bsonType: "int" },
        received_at:      { bsonType: "date" },
        updated_at:       { bsonType: "date" }
      }
    }
  },
  validationLevel: "strict",
  validationAction: "error"
});
```

### (b) `file_registry` — indexes

```javascript
// Duplicate detection: unique on checksum (or compound with source_id if dedup is per-source)
db.file_registry.createIndex({ sha256_checksum: 1 }, { unique: true });
db.file_registry.createIndex({ source_id: 1, received_at: -1 });
db.file_registry.createIndex({ current_status: 1 });
db.file_registry.createIndex({ file_type_id: 1, received_at: -1 });
db.file_registry.createIndex({ upload_session_id: 1 });
```

### (c) `event_log` — collection, validator, indexes

```javascript
db.createCollection("event_log", {
  validator: {
    $jsonSchema: {
      bsonType: "object",
      required: ["event_id", "event_type", "dedup_key", "received_at"],
      properties: {
        event_id:          { bsonType: "string", pattern: "^evt_" },
        event_type:       { enum: ["FILE_ARRIVED", "FILE_STAGED", "FILE_IDENTIFIED", "FILE_UNIDENTIFIED",
                                   "VALIDATION_COMPLETED", "FILE_ROUTED", "TRANSFER_COMPLETED",
                                   "TRANSFER_FAILED", "BRONZE_PROCESSED", "RECONCILIATION_COMPLETED",
                                   "FILE_ARCHIVED", "FILE_PURGED"] },
        dedup_key:        { bsonType: "string" },
        detection_method: { enum: ["PUSH", "WEBHOOK", "WATCHER", "POLL"] },
        source_id:        { bsonType: "string" },
        payload:          { bsonType: "object" },
        processing_status: { enum: ["RECEIVED", "PROCESSING", "PROCESSED", "DEAD"] },
        retry_count:      { bsonType: "int" },
        last_error:       { bsonType: ["string", "null"] },
        correlation_id:   { bsonType: ["string", "null"] },
        received_at:      { bsonType: "date" }
      }
    }
  },
  validationLevel: "strict",
  validationAction: "error"
});

// Duplicate deliveries are harmless:
db.event_log.createIndex({ event_type: 1, dedup_key: 1 }, { unique: true });

// Worker queue scan:
db.event_log.createIndex({ processing_status: 1, received_at: 1 });

// Ops view on DEAD events (re-drive):
db.event_log.createIndex({ received_at: 1 },
  { partialFilterExpression: { processing_status: "DEAD" }, name: "ix_event_log_dead" });

db.event_log.createIndex({ correlation_id: 1 });

// Optional: prune PROCESSED events after N days (never TTL on DEAD):
// db.event_log.createIndex({ received_at: 1 },
//   { expireAfterSeconds: 2592000,
//     partialFilterExpression: { processing_status: "PROCESSED" } });
```

> **Note:** change streams require a replica set (Atlas or self-managed).

### (d) `validation_rules` — collection, validator, indexes

```javascript
db.createCollection("validation_rules", {
  validator: {
    $jsonSchema: {
      bsonType: "object",
      required: ["rule_id", "rule_code", "category", "severity", "params", "is_active", "priority", "version"],
      properties: {
        rule_id:    { bsonType: "string", pattern: "^rule_" },
        rule_code:  { bsonType: "string" },
        category:   { enum: ["FILE", "HEADER", "CONTENT"] },
        severity:   { enum: ["ERROR", "WARN"] },
        applies_to: { bsonType: ["string", "null"] },   // file_type_id, or null = global
        params:     { bsonType: "object" },             // free-form rule parameters
        is_active:  { bsonType: "bool" },
        priority:   { bsonType: "int" },
        version:    { bsonType: "int" },
        created_by: { bsonType: "string" }
      }
    }
  },
  validationLevel: "strict",
  validationAction: "error"
});

db.validation_rules.createIndex({ rule_code: 1, version: -1 }, { unique: true });
db.validation_rules.createIndex({ is_active: 1, applies_to: 1, priority: 1 });
```

### (e) `schema_versions` — collection, validator, indexes

```javascript
db.createCollection("schema_versions", {
  validator: {
    $jsonSchema: {
      bsonType: "object",
      required: ["schema_id", "file_type_id", "version", "status", "fields"],
      properties: {
        schema_id:      { bsonType: "string", pattern: "^sch_" },
        file_type_id:   { bsonType: "string" },
        version:        { bsonType: "int" },
        status:         { enum: ["draft", "active", "retired"] },
        effective_from: { bsonType: "date" },
        fields: {
          bsonType: "array",
          minItems: 1,
          items: {
            bsonType: "object",
            required: ["order", "name"],
            properties: {
              order:              { bsonType: "int" },
              name:               { bsonType: "string" },
              data_type:          { bsonType: "string" },
              nullable:           { bsonType: "bool" },
              max_length:         { bsonType: ["int", "null"] },
              format:             { bsonType: ["string", "null"] },
              is_pii:             { bsonType: "bool" },
              pii_classification: { bsonType: ["string", "null"] }
            }
          }
        }
      }
    }
  },
  validationLevel: "strict",
  validationAction: "error"
});

db.schema_versions.createIndex({ file_type_id: 1, version: -1 }, { unique: true });
db.schema_versions.createIndex({ file_type_id: 1, status: 1, effective_from: -1 });
```

### (f) `file_transfers` — collection, validator, indexes

```javascript
db.createCollection("file_transfers", {
  validator: {
    $jsonSchema: {
      bsonType: "object",
      required: ["transfer_id", "file_id", "from_location", "to_location", "direction", "status", "attempts"],
      properties: {
        transfer_id:   { bsonType: "string", pattern: "^trf_" },
        file_id:       { bsonType: "string", pattern: "^file_" },
        policy_id:     { bsonType: ["string", "null"] },
        from_location: { bsonType: "string" },   // "volume:<staging_path>" (storage v1.3)
        to_location:   { bsonType: "string" },   // resolved volume path incl. subfolder
        direction:     { enum: ["MONGO_TO_INTAKE", "MONGO_TO_QUARANTINE", "MONGO_TO_ARCHIVE"] },  // legacy labels kept (Step-2 DDL + worker); volumes via location prefixes
        status:        { enum: ["PENDING", "IN_PROGRESS", "COMPLETED", "FAILED", "RETRYING"] },
        attempts:      { bsonType: "int", minimum: 0 },
        last_error:    { bsonType: ["string", "null"] },
        source_sha256: { bsonType: "string", minLength: 64, maxLength: 64 },
        dest_sha256:   { bsonType: ["string", "null"], minLength: 64, maxLength: 64 },
        bytes_written: { bsonType: ["long", "int", "null"] },
        worker_id:     { bsonType: ["string", "null"] },
        started_at:    { bsonType: ["date", "null"] },
        completed_at:  { bsonType: ["date", "null"] }
      }
    }
  },
  validationLevel: "strict",
  validationAction: "error"
});

db.file_transfers.createIndex({ file_id: 1, direction: 1, status: 1 });
db.file_transfers.createIndex({ status: 1, created_at: 1 });   // worker queue scan

// Idempotency: a file can never be pushed twice to the same destination class.
db.file_transfers.createIndex({ file_id: 1, direction: 1 },
  { unique: true, partialFilterExpression: { status: "COMPLETED" } });
```

### (g) `upload_sessions` — collection, validator, indexes

```javascript
db.createCollection("upload_sessions", {
  validator: {
    $jsonSchema: {
      bsonType: "object",
      required: ["session_id", "source_id", "user_id", "client_ip", "started_at"],
      properties: {
        session_id:         { bsonType: "string", pattern: "^ses_" },
        source_id:          { bsonType: "string" },
        user_id:            { bsonType: "string" },
        user_email:         { bsonType: ["string", "null"] },
        client_ip:          { bsonType: "string" },
        machine_name:       { bsonType: ["string", "null"] },
        user_agent:         { bsonType: ["string", "null"] },
        auth_method:        { bsonType: ["string", "null"] },
        roles:              { bsonType: "array", items: { bsonType: "string" } },
        files_declared_count: { bsonType: ["int", "null"] },
        files_received_count: { bsonType: "int" },
        started_at:         { bsonType: "date" },
        completed_at:       { bsonType: ["date", "null"] }
      }
    }
  },
  validationLevel: "strict",
  validationAction: "error"
});

db.upload_sessions.createIndex({ session_id: 1 }, { unique: true });
db.upload_sessions.createIndex({ user_id: 1, started_at: -1 });
```

### (h) `databricks_ingestion_results` — collection, validator, indexes

```javascript
db.createCollection("databricks_ingestion_results", {
  validator: {
    $jsonSchema: {
      bsonType: "object",
      required: ["ingestion_id", "file_id", "bronze_table", "rows_written", "processed_at"],
      properties: {
        ingestion_id:          { bsonType: "string" },
        run_id:                { bsonType: "string" },
        bronze_table:          { bsonType: "string" },
        file_type_id:          { bsonType: "string" },
        file_id:               { bsonType: "string", pattern: "^file_" },
        sha256:                { bsonType: "string", minLength: 64, maxLength: 64 },
        filename:              { bsonType: "string" },
        rows_read:             { bsonType: ["long", "int"] },
        rows_written:          { bsonType: ["long", "int"] },
        rows_quarantined_silver: { bsonType: ["long", "int"] },
        dq_results:            { bsonType: ["object", "null"] },
        processed_at:          { bsonType: "date" }
      }
    }
  },
  validationLevel: "strict",
  validationAction: "error"
});

db.databricks_ingestion_results.createIndex({ ingestion_id: 1 }, { unique: true });  // idempotent pushback
db.databricks_ingestion_results.createIndex({ file_id: 1, processed_at: -1 });
db.databricks_ingestion_results.createIndex({ run_id: 1 });
```

### (i) `reconciliation_results` — indexes

```javascript
db.reconciliation_results.createIndex({ run_id: 1 });
db.reconciliation_results.createIndex({ file_id: 1, run_id: 1 }, { unique: true });
db.reconciliation_results.createIndex({ status: 1, severity: 1 },
  { partialFilterExpression: { status: { $ne: "MATCHED" } } });   // open discrepancies only
```

### (j) `audit_log` — collection, validator, indexes

```javascript
db.createCollection("audit_log", {
  validator: {
    $jsonSchema: {
      bsonType: "object",
      required: ["event_id", "ts", "actor", "entity_type", "entity_id", "action"],
      properties: {
        event_id:      { bsonType: "string", pattern: "^aud_" },
        ts:            { bsonType: "date" },
        actor:         { bsonType: "string" },
        entity_type:   { bsonType: "string" },
        entity_id:     { bsonType: "string" },
        action:        { enum: ["CONFIG_CHANGE", "STATUS_CHANGE", "FILE_ROUTED", "RULE_ADDED", "OTHER"] },
        before:        { bsonType: ["object", "null"] },
        after:         { bsonType: ["object", "null"] },
        approver:      { bsonType: ["string", "null"] },
        correlation_id: { bsonType: ["string", "null"] }
      }
    }
  },
  validationLevel: "strict",
  validationAction: "error"
});

db.audit_log.createIndex({ entity_id: 1, ts: -1 });
db.audit_log.createIndex({ actor: 1, ts: -1 });
db.audit_log.createIndex({ correlation_id: 1 });
```

> **Append-only:** no updates — enforced via the application role, not schema.

### (k) `file_lifecycle` — indexes

```javascript
db.file_lifecycle.createIndex({ file_id: 1, timestamp: 1 });
```

### Conventions

* Pick **one** `_id` convention — prefixed strings (`file_`, `evt_`, `trf_`, …) or `ObjectId` — and use it consistently across collections.
* All timestamps are BSON dates in **UTC**.

## Databricks Ingestion-Results Pushback Job Design

**Purpose** — close the loop: per-file bronze/silver stats flow back into MongoDB `databricks_ingestion_results` so reconciliation can certify *uploaded = routed = transferred = bronze-processed*.

### File-level attribution

* Routing **renames files at transfer time** to `{file_type}_{file_id}_{ts}.ext` (`volume_map.naming_convention`), so bronze recovers `file_id` **directly from `_metadata.file_name`** — no join needed.
* `sha256` is carried as a bronze column, sourced from the transfer manifest.

### Implementation options

**OPTION A — streaming `foreachBatch` pushback (recommended, near-real-time):** the Auto Loader streaming job uses `foreachBatch`; after each micro-batch it aggregates per `_metadata.file_name` (`rows_read = count(*)`, `rows_written`) and POSTs one JSON payload per file to `FastAPI POST /internal/ingestion-results` via `httpx` with retry (3× exponential backoff) and a local DLQ table for failures.

**OPTION B — decoupled, batch:** a scheduled Lakeflow job (every 15–30 min) aggregates bronze per `_metadata` for files since the watermark into a UC table `config.ingestion_results_pushback`, and FastAPI polls that table. No inbound network path needed — at the cost of latency.

### Payload contract (matches `databricks_ingestion_results`)

```json
{
  "ingestion_id": "<file_id>|<bronze_table>|<run_id>",
  "run_id": "...",
  "bronze_table": "cat.schema.tbl",
  "file_type_id": "ft_customer",
  "file_id": "file_01HV3...",
  "sha256": "...",
  "filename": "customer_file_01HV3..._20261106T080000.csv",
  "rows_read": 10000,
  "rows_written": 10000,
  "rows_quarantined_silver": 0,
  "dq_results": {
    "rules_failed": [],
    "quarantined_rows": 0,
    "silver_status": "PASSED"
  },
  "processed_at": "2026-11-06T08:05:00Z"
}
```

### Security, idempotency, error handling

* **Security:** dedicated service principal + OAuth token, HTTPS; FastAPI authenticates via service token + allowlist — **never user tokens**.
* **Idempotency:** unique `ingestion_id` → pushback is an upsert; replays are harmless.
* **Silver DQ:** silver quarantine tables carry `_failed_rules`; pushback aggregates per source file via the `file_id` column propagated from bronze.
* **Error handling:** failed posts → DLQ + alert; pushback failures must **never fail the ingestion job** — isolate in a separate task so ingestion is not blocked by API issues.

### Pseudocode sketch (Option A — illustrative, not production)

```python
# PSEUDOCODE — Auto Loader foreachBatch pushback (PySpark + httpx)
from pyspark.sql import functions as F

def push_ingestion_results(batch_df, batch_id):
    # File-level attribution: filename is "{file_type}_{file_id}_{ts}.ext"
    stats = (
        batch_df
        .withColumn("file_name", F.col("_metadata.file_name"))
        .groupBy("file_name")
        .agg(
            F.count("*").alias("rows_read"),
            F.first("file_id").alias("file_id"),   # column propagated from transfer manifest
            F.first("sha256").alias("sha256"),
        )
        .collect()
    )

    payloads = [
        {
            "ingestion_id": f"{r.file_id}|{BRONZE_TABLE}|{batch_id}",
            "run_id": str(batch_id),
            "bronze_table": BRONZE_TABLE,
            "file_id": r.file_id,
            "sha256": r.sha256,
            "filename": r.file_name,
            "rows_read": r.rows_read,
            "rows_written": r.rows_read,             # adjust with silver DQ rollup when available
            "rows_quarantined_silver": 0,
            "dq_results": {"rules_failed": [], "quarantined_rows": 0, "silver_status": "PENDING"},
            "processed_at": utc_now_iso(),
        }
        for r in stats
    ]

    for p in payloads:
        try:
            post_with_retries(f"{FVR_API}/internal/ingestion-results", p)  # 3x exponential backoff
        except Exception:
            write_to_dlq(p)   # local Delta DLQ table — never fail the ingestion job
```

## Open Decisions / Next Steps

* **Event transport** — confirm MongoDB change streams vs RabbitMQ/Redis Streams based on expected event volume (change streams to start).
* **SharePoint** — Graph subscription management (renewal, validationToken handling) vs poll-only initially.
* **Archive target** — choose ADLS / S3 / archive volume for `retention_policies.archive_target`.
* **DLQ re-drive UX** — design the portal ops view + re-drive action for DEAD events and failed pushbacks.
* **Index strategy review** — re-check after the first load test.
* **Mongo deployment** — change streams require a replica set (Atlas or self-managed).

### Status update (2026-11-10)

**Resolved since the original open list:** storage = v1.3 direct-to-volume (GridFS deprecated; BSON-embedded BinData evaluated and declined — see the Physical Storage cell); volume roots = `mdd_platform.fvr.{staging,intake,quarantine}` created, seeded in `volume_map`, smoke-tested; event transport = change streams to start; Steps 1–4 authored and code-reviewed; dev SP `svc-fvr-dev` created (volume grants pending re-apply); pre-R2 dry-run 11/11 executable cases PASS on real volumes.

**Still open:** notification recipient emails (seed `notifications`); environment split for prod (SPs + volumes); rate tuning + locust S1–S5 (full R2 needs the deployed stack — see the Deployment Runbook notebook); post-R2 Databricks-side validation option (streaming table on intake / ack-driven trigger).